# 01 · Download Flymaster tasks to Google Drive

Downloads every task in `data/flymaster_catalog.csv` straight into Google Drive, without going through a laptop:

- `<task>.zip`: the IGC package (every pilot's track)
- `<task>.xctsk`: the task definition (turnpoints, cylinders, start, goal)
- `<task>_results.html`: the official results (rank, goal, times, glider, CIVL ID)

Files are saved in the project's Drive folder (`DRIVE_FOLDER_ID`), one subfolder per competition group. One request at a time, with a pause between requests.

**If Colab disconnects, just Run All again**: files already in Drive are skipped, so it picks up where it stopped.

## 1. Settings

In [ ]:
DRIVE_FOLDER_ID = "1_JO0XMMQ4NKVfpeA8uKQRkZZPNC4sqZ7"   # project folder in Google Drive (the id at the end of its link)
SUBFOLDER = "flymaster_igc"   # created inside that folder
PAUSE = 2.0          # seconds between requests
MIN_FREE_GB = 2.0    # stop when Drive has less free space than this
REPO_URL = "https://github.com/RobertojonesFAI/Pre-task-analyzer.git"
REPO = "/content/Pre-task-analyzer"

## 2. Mount Drive, find the project folder and get the code

The folder is found from its id, so it works whether it is in your own Drive or shared with you (if it was shared with you, add a shortcut to it in your Drive first: right click → Organize → Add shortcut).

In [ ]:
import os, shutil, subprocess
from google.colab import auth, drive
from googleapiclient.discovery import build

drive.mount("/content/drive")
auth.authenticate_user()
service = build("drive", "v3")


def drive_path(folder_id):
    """Local path of a Drive folder under /content/drive, from its id."""
    shortcut = f"/content/drive/.shortcut-targets-by-id/{folder_id}"
    if os.path.isdir(shortcut):
        return shortcut
    names, current = [], folder_id
    while True:
        meta = service.files().get(fileId=current, fields="name, parents", supportsAllDrives=True).execute()
        parents = meta.get("parents")
        if not parents:  # reached the root of My Drive
            return os.path.join("/content/drive/MyDrive", *reversed(names))
        names.append(meta["name"])
        current = parents[0]


ROOT = drive_path(DRIVE_FOLDER_ID)
assert os.path.isdir(ROOT), f"Drive folder not found at {ROOT}"
OUT = os.path.join(ROOT, SUBFOLDER)
os.makedirs(OUT, exist_ok=True)
print("Saving to", OUT)

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)

usage = shutil.disk_usage(OUT)
print(f"Drive: {usage.free / 1e9:.0f} GB free of {usage.total / 1e9:.0f} GB")

## 3. What will be downloaded

In [ ]:
import pandas as pd

catalog = pd.read_csv(f"{REPO}/data/flymaster_catalog.csv")
done = sum(os.path.exists(f"{OUT}/{g}/{t}.zip") for g, t in zip(catalog["group"], catalog["task"]))
print(f"{len(catalog)} tasks in {catalog['group'].nunique()} competitions, "
      f"{catalog['date'].min()} to {catalog['date'].max()}; {done} already in Drive")
catalog.head()

## 4. Download

Prints one line per task. Stop it any time with the stop button; rerun to continue.

In [ ]:
proc = subprocess.Popen(
    ["python3", "-u", f"{REPO}/scripts/download_flymaster.py", f"{REPO}/data/flymaster_catalog.csv", OUT,
     "--pause", str(PAUSE), "--min-free-gb", str(MIN_FREE_GB)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()

## 5. Check what is in Drive

In [ ]:
import zipfile
from pathlib import Path

zips = sorted(Path(OUT).glob("*/*.zip"))
tracks = 0
for z in zips:
    try:
        tracks += sum(name.lower().endswith(".igc") for name in zipfile.ZipFile(z).namelist())
    except zipfile.BadZipFile:
        print("bad zip:", z)
size = sum(z.stat().st_size for z in zips) / 1e9
print(f"{len(zips)} task packages, {tracks} IGC tracks, {size:.1f} GB")
print(f"{len(list(Path(OUT).glob('*/*.xctsk')))} task definitions, "
      f"{len(list(Path(OUT).glob('*/*_results.html')))} results pages")